Read video

In [1]:
import cv2

cap = cv2.VideoCapture("dynamic_raw/good_morning.mov")
frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)

cap.release()

print(f"Total frames in video: {len(frames)}")


Total frames in video: 244


Extract Landmarks
- Each norm_keypoints = flattened vector of 63 floats (21 points × 3).
- landmark_sequences = sequence of vectors for the video.

In [2]:
import mediapipe as mp
from fsl_preprocessing import extract_keypoints_from_hand_landmarks, normalize_landmarks


mp_draw = mp.solutions.drawing_utils

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)


landmark_sequences = []

for frame in frames:
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb_frame.flags.writeable = False
    result = hands.process(rgb_frame)
    rgb_frame.flags.writeable = True
    
    if result.multi_hand_landmarks:
        lm = result.multi_hand_landmarks[0]  # only first hand
        keypoints = extract_keypoints_from_hand_landmarks(lm)
        norm_keypoints = normalize_landmarks(keypoints, scale_mode='bbox')
        landmark_sequences.append(norm_keypoints)


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [4]:
import numpy as np


print("Sequence length:", len(landmark_sequences))
print("Landmark vector shape:", np.array(landmark_sequences).shape)

Sequence length: 119
Landmark vector shape: (119, 63)


Fix Sequence Length
- Now every video has shape (30, 63).
- Can be fed into LSTM.

In [5]:
import numpy as np
import importlib
import fsl_dynamic_utils
importlib.reload(fsl_dynamic_utils)
from fsl_dynamic_utils import pad_or_truncate_sequence


SEQ_LENGTH = 30  # all sequences will be 30 frames

# Check that there are actually frames
if len(landmark_sequences) == 0:
    raise ValueError("No landmarks detected in this video!")

# Apply padding/truncation
X_video = pad_or_truncate_sequence(landmark_sequences, length=SEQ_LENGTH)

# Convert to numpy array
X_video = np.array(X_video)  # shape: (30, 63)

# Optional debug
print("Final sequence shape:", X_video.shape)
print("Min / Max values:", X_video.min(), X_video.max())


Final sequence shape: (30, 63)
Min / Max values: -1.0 0.37636905718994595


In [8]:
mp_draw.draw_landmarks(frame, lm, mp.solutions.hands.HAND_CONNECTIONS)
cv2.imshow("Landmarks", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

### Train LSTM 


In [9]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [10]:
# -------------------------------
# 1️⃣ Configuration
# -------------------------------
VIDEO_DIR = "dynamic_raw"        # your dataset folder
SEQ_LENGTH = 30
SCALE_MODE = "bbox"

# Load CSV mapping: id -> label
csv_file = "csv/labels.csv"  # your CSV file
df = pd.read_csv(csv_file)
id_to_label = dict(zip(df["id"].astype(str), df["label"]))




In [11]:
# -------------------------------
# 2️⃣ Loop through selected folders (DEBUG MODE)
# -------------------------------

X_data = []
y_data = []

mp_hands = mp.solutions.hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5
)

subset_ids = ["0", "3", "7"]  # Only process these folders
# subset_ids = sorted(os.listdir(VIDEO_DIR))
MAX_VIDEOS_PER_CLASS = 5     # Limit to first 5 videos per sign, None if 

for folder_id in subset_ids:

    folder_path = os.path.join(VIDEO_DIR, folder_id)

    if not os.path.isdir(folder_path):
        print(f"Skipping non-directory: {folder_path}")
        continue

    # Get label from CSV
    if folder_id not in id_to_label:
        print(f"Warning: folder ID {folder_id} not in CSV")
        continue

    label = id_to_label[folder_id]

    video_files = sorted(os.listdir(folder_path))[:MAX_VIDEOS_PER_CLASS]

    print(f"\nProcessing class {label} (ID {folder_id})")

    for video_file in video_files:
        video_path = os.path.join(folder_path, video_file)

        cap = cv2.VideoCapture(video_path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            print(f"Warning: no frames in {video_path}")
            continue

        # Extract landmarks
        landmark_sequences = []

        for frame in frames:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = mp_hands.process(rgb_frame)

            if result.multi_hand_landmarks:
                lm = result.multi_hand_landmarks[0]
                keypoints = extract_keypoints_from_hand_landmarks(lm)
                norm_keypoints = normalize_landmarks(
                    keypoints,
                    scale_mode=SCALE_MODE
                )
                landmark_sequences.append(norm_keypoints)
            else:
                landmark_sequences.append(np.zeros(63).tolist())

        # Pad / truncate
        seq = pad_or_truncate_sequence(
            landmark_sequences,
            length=SEQ_LENGTH
        )

        X_data.append(seq)
        y_data.append(label)

        print(f"✔ Processed {video_file} → final shape: {len(seq)} frames")



Processing class GOOD MORNING (ID 0)


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✔ Processed 0.MOV → final shape: 30 frames
✔ Processed 1.MOV → final shape: 30 frames
✔ Processed 10.MOV → final shape: 30 frames
✔ Processed 11.MOV → final shape: 30 frames
✔ Processed 12.MOV → final shape: 30 frames

Processing class HELLO (ID 3)
✔ Processed 0.MOV → final shape: 30 frames
✔ Processed 1.MOV → final shape: 30 frames
✔ Processed 10.MOV → final shape: 30 frames
✔ Processed 11.MOV → final shape: 30 frames
✔ Processed 12.MOV → final shape: 30 frames

Processing class THANK YOU (ID 7)
✔ Processed 0.MOV → final shape: 30 frames
✔ Processed 1.MOV → final shape: 30 frames
✔ Processed 10.MOV → final shape: 30 frames
✔ Processed 11.MOV → final shape: 30 frames
✔ Processed 12.MOV → final shape: 30 frames


In [12]:
# -------------------------------
# 3️⃣ Convert to arrays
# -------------------------------
X_data = np.array(X_data)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)


In [13]:
# -------------------------------
# 4️⃣ Train / Val / Test Split
# -------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)


print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)


Train shape: (10, 30, 63) (10,)
Val shape: (2, 30, 63) (2,)
Test shape: (3, 30, 63) (3,)


In [14]:

# -------------------------------
# 5️⃣ Save artifacts
# -------------------------------
os.makedirs("models/dynamic", exist_ok=True)

np.save("models/dynamic/X_train.npy", X_train)
np.save("models/dynamic/X_val.npy", X_val)
np.save("models/dynamic/X_test.npy", X_test)
np.save("models/dynamic/y_train.npy", y_train)
np.save("models/dynamic/y_val.npy", y_val)
np.save("models/dynamic/y_test.npy", y_test)

with open("models/dynamic/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

preprocess_config = {
    "scale_mode": SCALE_MODE,
    "seq_length": SEQ_LENGTH
}

with open("models/dynamic/preprocess_config.pkl", "wb") as f:
    pickle.dump(preprocess_config, f)

print("Dynamic dataset processed and saved successfully!")


Dynamic dataset processed and saved successfully!


In [ ]:
import numpy as np
import pickle

# Load dynamic dataset
X_train = np.load("models/dynamic/X_train.npy")
y_train = np.load("models/dynamic/y_train.npy")

with open("models/dynamic/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Number of classes:", len(le.classes_))
print("Classes:", le.classes_)


print("One sample shape:", X_train[0].shape)
print("Min value:", X_train.min())
print("Max value:", X_train.max())



In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

num_classes = len(le.classes_)

model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(30, 63)))
model.add(Dropout(0.3))
model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(64))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=8,
    callbacks=[early_stop]
)



c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 432ms/step - accuracy: 0.4333 - loss: 1.0988 - val_accuracy: 0.0000e+00 - val_loss: 1.0993
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.3917 - loss: 1.0985 - val_accuracy: 0.0000e+00 - val_loss: 1.0984
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3917 - loss: 1.0986 - val_accuracy: 0.0000e+00 - val_loss: 1.0977
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.3917 - loss: 1.0985 - val_accuracy: 0.0000e+00 - val_loss: 1.0972
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.3917 - loss: 1.0985 - val_accuracy: 0.0000e+00 - val_loss: 1.0968
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4333 - loss: 1.0983 - val_accuracy: 0.0000e+00 - val_loss: 1.0964
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.3500 - loss: 1.0985 - val_accuracy: 0.0000e+00 - val_loss: 1.0960
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.4333 - loss: 1.0981 - val

In [ ]:
# model = Sequential() # full ver

# model.add(LSTM(128, return_sequences=True, input_shape=(30, 63)))
# model.add(Dropout(0.3))
# model.add(LSTM(256, return_s  equences=True))
# model.add(Dropout(0.3))
# model.add(LSTM(128))
# model.add(Dropout(0.3))

# model.add(Dense(256, activation='relu'))
# model.add(Dense(num_classes, activation='softmax'))

In [16]:
from collections import Counter
print("Class distribution:", Counter(y_train))


Class distribution: Counter({np.int64(0): 4, np.int64(1): 3, np.int64(2): 3})


In [17]:
model.save("models/dynamic/subset_lstm_model.h5")

In [18]:
print("Sequence shape:", np.array(sequence).shape)


NameError: name 'sequence' is not defined